# PCCP / FCP Verification on NASA Li-ion Battery RUL  (Colab GPU) — v4

Cross-domain validation of **Theorems 1, 2, 3, 5**, **Proposition 1**, and **Corollary 5.1** of the
Physics-Constrained Conformal Prediction (PCCP) paper, on the NASA Li-ion battery aging dataset.

## What is new in v4

- **Subgroup coverage on battery (Theorem 2 test).** For each test cell, we compute PICP within the safety-critical subgroups $\mathrm{RUL} < 30$ and $\mathrm{RUL} < 15$, then verify Theorem 2's prediction that the subgroup PICP of $C^{\mathrm{proj}}$ equals that of $C$ in every fold. The previous versions verified Theorem 2 on C-MAPSS only; v4 closes that gap on the battery side.
- **Empirical surrogate for Theorem 5 (KS-distance).** For each fold, we compute the Kolmogorov--Smirnov distance between calibration scores (from the two training cells) and test scores (from the held-out cell). KS distance is a 1-D lower bound on the TV distance that appears in Theorem 5's bound $\mathrm{PICP} \geq 1 - \alpha - \varepsilon$; we compare the empirical KS magnitude per cell to the observed coverage gap as a consistency check for Theorem 5.
- **Per-test-point bookkeeping.** `run_one_fold` now returns the test labels and interval bounds and the score arrays per fold, so that the new analyses (and any audit-style downstream analysis) can be performed without rerunning the GPU pipeline.

All other components match v3: 3 valid cells (B0007 right-censored, excluded), 50 seeds, 150 folds, $\Rmax^{\mathrm{batt}} = 124$, general-form wasted budget $\widehat{\mathcal{W}}$ (Eq.\ 7).

## Runtime
On Colab T4 GPU: ~10 minutes total. The new analyses run in seconds (no GPU work).

## Data source
https://data.nasa.gov/dataset/li-ion-battery-aging-datasets — `B0005.mat`, `B0006.mat`, `B0007.mat`, `B0018.mat`.


## 1. Setup


In [ ]:
# Verify GPU + PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Standard imports
import os, json, warnings
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from typing import Tuple, Dict, List
from tqdm import tqdm
from scipy.io import loadmat
from scipy.stats import ks_2samp  # NEW in v4: for Theorem 5 KS surrogate
warnings.filterwarnings("ignore")


## 2. Upload battery data

Upload `B0005.mat`, `B0006.mat`, `B0007.mat`, `B0018.mat` via the Colab file browser
(left sidebar -> folder icon -> upload), or use the upload widget below.


In [ ]:
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    print(f"  Uploaded: {fname}  ({len(uploaded[fname]):,} bytes)")


In [ ]:
expected = ["B0005.mat", "B0006.mat", "B0007.mat", "B0018.mat"]
missing = [f for f in expected if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")
print("All four cell files present.")


## 3. Per-cycle feature extraction


In [ ]:
def load_cell_discharges(matfile: str) -> pd.DataFrame:
    mat = loadmat(matfile, squeeze_me=True, struct_as_record=False)
    cell_name = os.path.splitext(os.path.basename(matfile))[0]
    cell_struct = mat[cell_name]
    cycles = cell_struct.cycle
    rows = []
    discharge_idx = 0
    for c in cycles:
        if c.type != "discharge":
            continue
        d = c.data
        V = np.asarray(d.Voltage_measured).flatten()
        I = np.asarray(d.Current_measured).flatten()
        T = np.asarray(d.Temperature_measured).flatten()
        t = np.asarray(d.Time).flatten()
        cap = float(np.asarray(d.Capacity).flatten()[0])
        if len(V) < 5:
            continue
        def slope(y):
            x = np.arange(len(y))
            return float(np.polyfit(x, y, 1)[0])
        row = {
            "cycle_idx": discharge_idx, "capacity": cap,
            "discharge_time": float(t[-1] - t[0]),
            "V_mean": float(V.mean()), "V_std": float(V.std()),
            "V_min": float(V.min()), "V_max": float(V.max()),
            "V_slope": slope(V),
            "I_mean": float(I.mean()), "I_std": float(I.std()),
            "T_mean": float(T.mean()), "T_max": float(T.max()),
            "T_std": float(T.std()), "T_slope": slope(T),
        }
        rows.append(row); discharge_idx += 1
    df = pd.DataFrame(rows); df["cell"] = cell_name
    return df

cells_raw = {}
for f in expected:
    df = load_cell_discharges(f)
    cells_raw[df["cell"].iloc[0]] = df
    print(f"  {df['cell'].iloc[0]}: {len(df)} discharge cycles, "
          f"capacity {df['capacity'].iloc[0]:.3f} -> {df['capacity'].iloc[-1]:.3f} Ah")


In [ ]:
EOL_CAPACITY = 1.4

def compute_rul_strict(df: pd.DataFrame):
    df = df.copy().reset_index(drop=True)
    eol_mask = df["capacity"] <= EOL_CAPACITY
    if eol_mask.any():
        T_e = int(df.index[eol_mask][0])
        df["RUL"] = T_e - df["cycle_idx"]
        df["RUL"] = df["RUL"].clip(lower=0).astype(int)
        return df, True, T_e
    else:
        df["RUL"] = np.nan
        return df, False, None

valid_cells = []
censored_cells = []
for name in cells_raw:
    df_with_rul, reached, T_e = compute_rul_strict(cells_raw[name])
    if reached:
        valid_cells.append(name)
        cells_raw[name] = df_with_rul
        print(f"  {name}: EOL at cycle {T_e}, max RUL = {int(df_with_rul['RUL'].max())} (valid)")
    else:
        censored_cells.append(name)
        print(f"  {name}: NEVER reaches EOL, EXCLUDED (right-censored)")

print()
print(f"Valid cells:    {valid_cells}")
print(f"Censored cells: {censored_cells}")

cells_df = {name: cells_raw[name] for name in valid_cells}
assert all(cells_df[c]["RUL"].notna().all() for c in cells_df)


In [ ]:
R_MAX = int(max(df["RUL"].max() for df in cells_df.values()))
print(f"R_max (battery v4 valid cohort) = {R_MAX} cycles")
max_test = max(df["RUL"].max() for df in cells_df.values())
min_test = min(df["RUL"].min() for df in cells_df.values())
assert min_test >= 0 and max_test <= R_MAX
print(f"Sanity check passed: all RUL labels in [0, {R_MAX}].")


## 4. MLP base predictor


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64, 32)):
        super().__init__()
        layers = []; prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp(X_tr, y_tr, X_val, y_val, in_dim, epochs=80, lr=1e-3,
              batch_size=256, device="cuda"):
    model = MLP(in_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    ds = TensorDataset(
        torch.as_tensor(X_tr, dtype=torch.float32, device=device),
        torch.as_tensor(y_tr, dtype=torch.float32, device=device),
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    Xv = torch.as_tensor(X_val, dtype=torch.float32, device=device)
    yv = torch.as_tensor(y_val, dtype=torch.float32, device=device)
    best_state, best_val = None, float("inf")
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v = float(crit(model(Xv), yv).item())
            if v < best_val:
                best_val = v
                best_state = {k: x.clone() for k, x in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def predict(model, X, device="cuda"):
    model.eval()
    with torch.no_grad():
        return model(torch.as_tensor(X, dtype=torch.float32, device=device)).cpu().numpy()


## 5. CP / PCCP / FCP definitions


In [ ]:
def project_to_K(y, lo=0.0, hi=None):
    if hi is None:
        return np.maximum(y, lo)
    return np.clip(y, lo, hi)

def split_cp_quantile(scores, alpha):
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    k = min(max(k, 1), n)
    return float(np.sort(scores)[k - 1])

def cp_intervals(f_test, q_alpha, lo=0.0, hi=None):
    L = f_test - q_alpha; U = f_test + q_alpha
    L_proj = np.maximum(L, lo) if lo is not None else L
    U_proj = np.minimum(U, hi) if hi is not None else U
    return L, U, L_proj, U_proj

def wasted_budget_general(L, U, R_max):
    """General form (Eq 7), holds for any interval including those entirely outside K."""
    if R_max is None:
        return np.maximum(0, -L)
    lower = np.maximum(0, np.minimum(U, 0) - L)
    upper = np.maximum(0, U - np.maximum(L, R_max))
    return lower + upper

def wasted_budget_simplified(L, U, R_max):
    if R_max is None:
        return np.maximum(0, -L)
    return np.maximum(0, -L) + np.maximum(0, U - R_max)

def metrics(y_test, L, U, R_max=None):
    cov = ((y_test >= L) & (y_test <= U)).astype(float)
    width = np.maximum(U - L, 0.0)
    psi_neg = (L < 0).astype(float).mean()
    psi_pos = (U > R_max).astype(float).mean() if R_max is not None else 0.0
    psi_ok = ((L >= 0) & ((U <= R_max) if R_max is not None else True)).astype(float).mean()
    waste_gen = wasted_budget_general(L, U, R_max).mean()
    waste_simp = wasted_budget_simplified(L, U, R_max).mean()
    return {
        "PICP": cov.mean(), "MPIW": width.mean(),
        "psi_neg": psi_neg * 100, "psi_pos": psi_pos * 100, "psi_ok": psi_ok * 100,
        "wasted": waste_gen, "wasted_simplified": waste_simp,
    }


## 6. Single-fold runner (v4: returns per-test-point arrays for downstream analysis)

v4 augments v3 by also returning:
- `y_test`, `L_cp`, `U_cp`, `L_proj`, `U_proj`, `L_fcp`, `U_fcp`: per-test-point arrays for subgroup analysis (Theorem 2).
- `s_cal`, `s_test`: per-fold calibration and test score arrays for KS-distance computation (Theorem 5 surrogate).


In [ ]:
FEATURE_COLS = [
    "discharge_time",
    "V_mean", "V_std", "V_min", "V_max", "V_slope",
    "I_mean", "I_std",
    "T_mean", "T_max", "T_std", "T_slope",
    "capacity",
]

def run_one_fold(test_cell: str, seed: int, alpha: float = 0.1, R_max: int = None):
    np.random.seed(seed); torch.manual_seed(seed)
    train_cells = [c for c in cells_df if c != test_cell]
    df_tr = pd.concat([cells_df[c] for c in train_cells], ignore_index=True)
    df_te = cells_df[test_cell].copy()

    idx = np.arange(len(df_tr)); np.random.shuffle(idx)
    n_train = int(0.7 * len(df_tr))
    train_idx, cal_idx = idx[:n_train], idx[n_train:]
    df_train = df_tr.iloc[train_idx].reset_index(drop=True)
    df_cal   = df_tr.iloc[cal_idx].reset_index(drop=True)

    X_train = df_train[FEATURE_COLS].values.astype(np.float32)
    y_train = df_train["RUL"].values.astype(np.float32)
    X_cal   = df_cal[FEATURE_COLS].values.astype(np.float32)
    y_cal   = df_cal["RUL"].values.astype(np.float32)
    X_test  = df_te[FEATURE_COLS].values.astype(np.float32)
    y_test  = df_te["RUL"].values.astype(np.float32)

    mu, sd = X_train.mean(0), X_train.std(0) + 1e-8
    X_train = (X_train - mu) / sd; X_cal = (X_cal - mu) / sd; X_test = (X_test - mu) / sd

    n_val = max(int(0.1 * len(X_train)), 5)
    val_idx_local = np.random.permutation(len(X_train))[:n_val]
    train_mask = np.ones(len(X_train), dtype=bool); train_mask[val_idx_local] = False
    Xtr_, ytr_ = X_train[train_mask], y_train[train_mask]
    Xval_, yval_ = X_train[~train_mask], y_train[~train_mask]

    model = train_mlp(Xtr_, ytr_, Xval_, yval_, in_dim=X_train.shape[1], device=DEVICE)

    f_cal  = predict(model, X_cal,  device=DEVICE)
    f_test = predict(model, X_test, device=DEVICE)
    s_cp = np.abs(y_cal - f_cal)
    s_test_cp = np.abs(y_test - f_test)  # NEW v4: test-side scores for KS
    q_cp = split_cp_quantile(s_cp, alpha)

    ftil_cal  = project_to_K(f_cal,  0.0, R_max)
    ftil_test = project_to_K(f_test, 0.0, R_max)
    s_fcp = np.abs(y_cal - ftil_cal)
    q_fcp = split_cp_quantile(s_fcp, alpha)

    L_cp, U_cp, L_proj, U_proj = cp_intervals(f_test, q_cp, 0.0, R_max)
    L_fcp_raw = ftil_test - q_fcp; U_fcp_raw = ftil_test + q_fcp
    L_fcp = np.maximum(L_fcp_raw, 0.0)
    U_fcp = np.minimum(U_fcp_raw, R_max) if R_max is not None else U_fcp_raw

    return {
        "test_cell": test_cell, "seed": seed,
        "n_train": len(X_train), "n_cal": len(X_cal), "n_test": len(X_test),
        "q_cp": q_cp, "q_fcp": q_fcp,
        "delta_pred_test": float(np.mean(np.abs(f_test - ftil_test))),
        "cp":   metrics(y_test, L_cp,   U_cp,   R_max=R_max),
        "pccp": metrics(y_test, L_proj, U_proj, R_max=R_max),
        "fcp":  metrics(y_test, L_fcp,  U_fcp,  R_max=R_max),
        # NEW in v4: per-test-point arrays for subgroup + KS analyses
        "y_test": y_test, "L_cp": L_cp, "U_cp": U_cp,
        "L_proj": L_proj, "U_proj": U_proj,
        "L_fcp": L_fcp, "U_fcp": U_fcp,
        "s_cal": s_cp, "s_test": s_test_cp,
    }


## 7. Run all folds


In [ ]:
N_SEEDS = 50
ALPHA = 0.1

results: List[Dict] = []
for cell_name in valid_cells:
    for seed in tqdm(range(N_SEEDS), desc=cell_name):
        try:
            out = run_one_fold(cell_name, seed, alpha=ALPHA, R_max=R_MAX)
            results.append(out)
        except Exception as e:
            print(f"  fold ({cell_name}, seed {seed}) FAILED: {e}")
print(f"Total successful folds: {len(results)}  (expected {len(valid_cells) * N_SEEDS})")


## 8. Aggregate to Table 6 of paper


In [ ]:
def aggregate_per_cell(results, cell):
    rows = [r for r in results if r["test_cell"] == cell]
    def avg(method, key):
        vals = [r[method][key] for r in rows]
        return float(np.mean(vals)), float(np.std(vals))
    out = {}
    for method in ["cp", "pccp", "fcp"]:
        out[method] = {
            "PICP_mean": avg(method, "PICP")[0], "PICP_std": avg(method, "PICP")[1],
            "MPIW_mean": avg(method, "MPIW")[0], "MPIW_std": avg(method, "MPIW")[1],
            "psi_neg":   avg(method, "psi_neg")[0],
            "psi_pos":   avg(method, "psi_pos")[0],
            "psi_ok":    avg(method, "psi_ok")[0],
            "wasted":    avg(method, "wasted")[0],
        }
    return out

table_rows = []
for cell in valid_cells:
    a = aggregate_per_cell(results, cell)
    for method in ["cp", "pccp", "fcp"]:
        m = a[method]
        table_rows.append({
            "Cell": cell, "Method": method.upper(),
            "PICP": f"{m['PICP_mean']:.3f} ({m['PICP_std']:.3f})",
            "MPIW": f"{m['MPIW_mean']:.2f}",
            "psi_neg (%)": f"{m['psi_neg']:.1f}",
            "psi_pos (%)": f"{m['psi_pos']:.1f}",
            "psi_ok (%)":  f"{m['psi_ok']:.1f}",
        })
for method in ["cp", "pccp", "fcp"]:
    avg_pi = float(np.mean([r[method]['PICP'] for r in results]))
    avg_pi_std = float(np.std([r[method]['PICP'] for r in results]))
    avg_mp = float(np.mean([r[method]['MPIW'] for r in results]))
    avg_neg = float(np.mean([r[method]['psi_neg'] for r in results]))
    avg_pos = float(np.mean([r[method]['psi_pos'] for r in results]))
    avg_ok = float(np.mean([r[method]['psi_ok'] for r in results]))
    table_rows.append({
        "Cell": "Average", "Method": method.upper(),
        "PICP": f"{avg_pi:.3f} ({avg_pi_std:.3f})",
        "MPIW": f"{avg_mp:.2f}",
        "psi_neg (%)": f"{avg_neg:.1f}",
        "psi_pos (%)": f"{avg_pos:.1f}",
        "psi_ok (%)":  f"{avg_ok:.1f}",
    })
table_df = pd.DataFrame(table_rows)
print("=" * 80)
print(f"TABLE 6 (v4): Battery RUL cross-domain validation, {len(valid_cells)} valid cells, {len(results)} folds")
print("=" * 80)
print(table_df.to_string(index=False))


## 9. Verify Proposition 1 / Corollary 5.1 on battery data


In [ ]:
discrepancies_gen = []
for r in results:
    mpiw_diff = r["cp"]["MPIW"] - r["pccp"]["MPIW"]
    waste = r["cp"]["wasted"]
    discrepancies_gen.append(abs(mpiw_diff - waste))
discrepancies_gen = np.array(discrepancies_gen)
print("Per-fold |MPIW(CP) - MPIW(PCCP) - W_general(C,K)|:  (theorem check)")
print(f"  max = {discrepancies_gen.max():.10f}, mean = {discrepancies_gen.mean():.10f}, median = {np.median(discrepancies_gen):.10f}")
print()
gaps = []
for r in results:
    gen = r["cp"]["wasted"]; simp = r["cp"]["wasted_simplified"]
    gaps.append(abs(gen - simp))
gaps = np.array(gaps)
print("Per-fold |W_general - W_simplified|:")
print(f"  max = {gaps.max():.6f}, mean = {gaps.mean():.6f}, median = {np.median(gaps):.6f}")
n_simp_valid = int((gaps < 1e-9).sum())
print(f"  Folds where simplified == general (within 1e-9): {n_simp_valid}/{len(gaps)}")


## 10. Verify Theorem 3 parts (i)-(iv) on battery data


In [ ]:
qtil_le_qhat = sum(1 for r in results if r["q_fcp"] <= r["q_cp"] + 1e-9)
print(f"Theorem 3 (iii) Quantile inequality q_tilde <= q_hat:")
print(f"  Holds in {qtil_le_qhat} / {len(results)} folds = {100*qtil_le_qhat/len(results):.1f}%")

n_valid_cp = sum(1 for r in results if r["cp"]["PICP"] >= 0.85)
n_valid_pccp = sum(1 for r in results if r["pccp"]["PICP"] >= 0.85)
print(f"\nTheorem 3 (i) Validity (PICP >= 0.85):")
print(f"  CP:   {n_valid_cp}/{len(results)}")
print(f"  PCCP: {n_valid_pccp}/{len(results)}")

simple_holds = 0; full_holds = 0
for r in results:
    gap = abs(r["pccp"]["MPIW"] - r["fcp"]["MPIW"])
    simple_b = 2 * (r["q_cp"] - r["q_fcp"])
    full_b = r["delta_pred_test"] + simple_b
    if gap <= simple_b + 1e-9: simple_holds += 1
    if gap <= full_b + 1e-9: full_holds += 1
print(f"\nTheorem 3 (iv) bound without Delta_pred (simpler form):")
print(f"  Holds in {simple_holds} / {len(results)} folds = {100*simple_holds/len(results):.1f}%")
print(f"\nTheorem 3 (iv) bound including Delta_pred (Eq. 12, full form):")
print(f"  Holds in {full_holds} / {len(results)} folds = {100*full_holds/len(results):.1f}%")


## 11. Subgroup coverage on battery (Theorem 2 test, NEW in v4)

Theorem 2 states that conditional coverage on every measurable subgroup is preserved exactly by the projection: for any subgroup $G \subseteq \X$,
$$\Prob(Y \in C^{\mathrm{proj}}(X) \mid X \in G) = \Prob(Y \in C(X) \mid X \in G).$$

For C-MAPSS we verified this on subgroups $\{\mathrm{RUL} < 50, 30, 15\}$ (Table 2). Here we verify the same property on battery, with subgroups $\{\mathrm{RUL} < 30, 15\}$ (the canonical near-end-of-life regions on this dataset's RUL scale of $[0, 124]$).

For every fold, the empirical event-set identity $\mathbf{1}\{Y \in C \cap \Kc\} = \mathbf{1}\{Y \in C\}$ at every test point implies the per-subgroup PICP also coincides exactly. The check below confirms this empirically.


In [ ]:
def subgroup_picp(y_test, L, U, threshold):
    """PICP within the subgroup {y_test < threshold}, NaN if subgroup is empty."""
    mask = y_test < threshold
    if mask.sum() == 0:
        return float("nan"), 0
    cov = ((y_test[mask] >= L[mask]) & (y_test[mask] <= U[mask])).mean()
    return float(cov), int(mask.sum())

def subgroup_mpiw(L, U, y_test, threshold):
    mask = y_test < threshold
    if mask.sum() == 0:
        return float("nan")
    return float(np.maximum(U[mask] - L[mask], 0.0).mean())

thresholds = [30, 15]

# Per-fold subgroup metrics
subgroup_records = []
for r in results:
    y = r["y_test"]
    for tau in thresholds:
        picp_cp,    n_sub = subgroup_picp(y, r["L_cp"],   r["U_cp"],   tau)
        picp_pccp,  _     = subgroup_picp(y, r["L_proj"], r["U_proj"], tau)
        picp_fcp,   _     = subgroup_picp(y, r["L_fcp"],  r["U_fcp"],  tau)
        mpiw_cp   = subgroup_mpiw(r["L_cp"],   r["U_cp"],   y, tau)
        mpiw_pccp = subgroup_mpiw(r["L_proj"], r["U_proj"], y, tau)
        mpiw_fcp  = subgroup_mpiw(r["L_fcp"],  r["U_fcp"],  y, tau)
        subgroup_records.append({
            "test_cell": r["test_cell"], "seed": r["seed"], "tau": tau, "n_sub": n_sub,
            "picp_cp": picp_cp, "picp_pccp": picp_pccp, "picp_fcp": picp_fcp,
            "mpiw_cp": mpiw_cp, "mpiw_pccp": mpiw_pccp, "mpiw_fcp": mpiw_fcp,
        })
sub_df = pd.DataFrame(subgroup_records)

# Theorem 2 verification: per-fold equality of subgroup PICP for CP vs PCCP
print("=" * 78)
print("Theorem 2 verification on battery: PICP_subgroup(C^proj) == PICP_subgroup(C)")
print("=" * 78)
for tau in thresholds:
    sub = sub_df[(sub_df["tau"] == tau) & sub_df["picp_cp"].notna()]
    n_total = len(sub)
    n_eq = int((np.abs(sub["picp_cp"] - sub["picp_pccp"]) < 1e-12).sum())
    print(f"  RUL < {tau}: PICP_CP == PICP_PCCP in {n_eq}/{n_total} folds (where subgroup non-empty)")

# Per-cell aggregation
print()
print("=" * 78)
print("Per-cell subgroup statistics (mean over seeds; NaN if subgroup empty)")
print("=" * 78)
agg_rows = []
for cell in valid_cells:
    for tau in thresholds:
        sub = sub_df[(sub_df["test_cell"] == cell) & (sub_df["tau"] == tau)
                     & sub_df["picp_cp"].notna()]
        if len(sub) == 0:
            agg_rows.append({"Cell": cell, "tau": tau, "n_seeds": 0,
                             "picp_cp": float("nan"), "picp_pccp": float("nan"),
                             "mpiw_cp": float("nan"), "mpiw_pccp": float("nan")})
            continue
        agg_rows.append({
            "Cell": cell, "tau": tau, "n_seeds": len(sub),
            "n_sub_avg": float(sub["n_sub"].mean()),
            "picp_cp":   float(sub["picp_cp"].mean()),
            "picp_pccp": float(sub["picp_pccp"].mean()),
            "mpiw_cp":   float(sub["mpiw_cp"].mean()),
            "mpiw_pccp": float(sub["mpiw_pccp"].mean()),
        })
agg_df = pd.DataFrame(agg_rows)
with pd.option_context("display.float_format", "{:.3f}".format):
    print(agg_df.to_string(index=False))

# Save for paper
sub_df.to_csv("battery_subgroup_per_fold_v4.csv", index=False)
agg_df.to_csv("battery_subgroup_per_cell_v4.csv", index=False)
print("\nSaved battery_subgroup_per_fold_v4.csv and battery_subgroup_per_cell_v4.csv")


## 12. Empirical Theorem 5 surrogate: KS-distance and coverage gap (NEW in v4)

Theorem 5 states $\mathrm{PICP}(\Cproj) \geq 1 - \alpha - \varepsilon$ where $\varepsilon = d_{\mathrm{TV}}(Q, P^{\otimes(n+1)})$ is the TV-distance between the actual joint distribution and the exchangeable one. Computing $d_{\mathrm{TV}}$ exactly on a high-dimensional joint is intractable; we use the **Kolmogorov--Smirnov distance between calibration scores and test scores** as a 1-D distributional surrogate. KS distance is a lower bound on TV distance for the score marginals, so the consistency check is:

$$ \underbrace{1 - \alpha - \mathrm{PICP}_{\text{test}}}_{\text{observed coverage gap}} \;\lesssim\; \underbrace{d_{\mathrm{TV}}(Q, P^{\otimes(n+1)})}_{\text{Theorem 5 bound}} \;\geq\; \underbrace{d_{\mathrm{KS}}(s_{\text{cal}}, s_{\text{test}})}_{\text{this surrogate}} $$

That is, the observed coverage gap is at most the TV distance, and the TV distance is at least the KS distance on score marginals. If the KS distance is of comparable order to the coverage gap, the data is consistent with Theorem 5; if KS is much smaller than the coverage gap, the bound would still hold (TV $\geq$ KS), but the score marginals alone would not explain the gap.

This is a **consistency check**, not a tight verification: KS only sees the score marginals, not the full joint, so we expect KS to be a loose lower bound on TV.


In [ ]:
# Per-fold KS distance between calibration scores and test scores
ks_records = []
for r in results:
    ks_stat, ks_pval = ks_2samp(r["s_cal"], r["s_test"])
    cov_gap_cp   = max(0.0, (1 - ALPHA) - r["cp"]["PICP"])
    cov_gap_pccp = max(0.0, (1 - ALPHA) - r["pccp"]["PICP"])
    ks_records.append({
        "test_cell": r["test_cell"], "seed": r["seed"],
        "n_cal": len(r["s_cal"]), "n_test": len(r["s_test"]),
        "ks_distance": float(ks_stat), "ks_pvalue": float(ks_pval),
        "coverage_gap_cp":   cov_gap_cp,
        "coverage_gap_pccp": cov_gap_pccp,
    })
ks_df = pd.DataFrame(ks_records)

print("=" * 80)
print("KS-distance vs coverage-gap per cell (Theorem 5 consistency check)")
print("=" * 80)
for cell in valid_cells:
    sub = ks_df[ks_df["test_cell"] == cell]
    ks_mean = sub["ks_distance"].mean()
    ks_std  = sub["ks_distance"].std()
    gap_cp  = sub["coverage_gap_cp"].mean()
    gap_pccp = sub["coverage_gap_pccp"].mean()
    print(f"  {cell}: KS = {ks_mean:.3f} ({ks_std:.3f}),  "
          f"coverage gap (CP) = {gap_cp:.3f},  coverage gap (PCCP) = {gap_pccp:.3f}")

print()
all_ks_mean = ks_df["ks_distance"].mean()
all_ks_std  = ks_df["ks_distance"].std()
all_gap_cp  = ks_df["coverage_gap_cp"].mean()
all_gap_pccp = ks_df["coverage_gap_pccp"].mean()
print(f"  Average over 150 folds:")
print(f"    KS distance:        {all_ks_mean:.3f} ({all_ks_std:.3f})")
print(f"    Coverage gap (CP):  {all_gap_cp:.3f}")
print(f"    Coverage gap (PCCP):{all_gap_pccp:.3f}")

# Theorem 5 statement check: PICP >= 1 - alpha - eps
# Empirically: cov_gap = 1 - alpha - PICP <= TV >= KS
# So: cov_gap <= TV is the theorem, KS is a lower bound on TV.
# We check: per-fold cov_gap and per-fold KS magnitudes are of comparable order.
print()
print("=" * 80)
print("Per-fold consistency: how often is the observed coverage gap <= KS?")
print("  (KS is a LOWER bound on TV, so cov_gap <= KS is sufficient but not necessary;")
print("   cov_gap > KS still allows Theorem 5 to hold if TV > KS, which is typical.)")
print("=" * 80)
n_below_ks = int((ks_df["coverage_gap_cp"] <= ks_df["ks_distance"]).sum())
print(f"  cov_gap_cp <= KS:   {n_below_ks}/{len(ks_df)} folds")

# Save
ks_df.to_csv("battery_ks_per_fold_v4.csv", index=False)
print("\nSaved battery_ks_per_fold_v4.csv")


## 13. Save raw results


In [ ]:
table_df.to_csv("battery_table6_v4.csv", index=False)
print("Saved battery_table6_v4.csv")

out_rows = []
for r in results:
    base = {"test_cell": r["test_cell"], "seed": r["seed"],
            "q_cp": r["q_cp"], "q_fcp": r["q_fcp"],
            "delta_pred_test": r["delta_pred_test"]}
    for method in ["cp", "pccp", "fcp"]:
        for key, val in r[method].items():
            base[f"{method}_{key}"] = val
    out_rows.append(base)
pd.DataFrame(out_rows).to_csv("battery_per_fold_v4.csv", index=False)
print("Saved battery_per_fold_v4.csv")

# Download all v4 outputs
from google.colab import files as colab_files
for f in ["battery_table6_v4.csv", "battery_per_fold_v4.csv",
          "battery_subgroup_per_fold_v4.csv", "battery_subgroup_per_cell_v4.csv",
          "battery_ks_per_fold_v4.csv"]:
    try:
        colab_files.download(f)
    except Exception as e:
        print(f"Could not download {f}: {e}")


## 14. LaTeX rows for Table 6


In [ ]:
def latex_row(cell, method, m, bold=False):
    method_str = f"\\textbf{{{method.upper()}}}" if bold else method.upper()
    cell_str = f"\\textbf{{{cell}}}" if cell == "Average" else cell
    picp_str = f"{m['PICP_mean']:.3f} ({m['PICP_std']:.3f})"
    mpiw_str = f"{m['MPIW_mean']:.2f}"
    if bold:
        mpiw_str = f"\\textbf{{{mpiw_str}}}"
        picp_str = f"\\textbf{{{picp_str}}}"
    return (f"{cell_str} & {method_str} & {picp_str} & {mpiw_str} & "
            f"{m['psi_neg']:.1f} & {m['psi_pos']:.1f} & {m['psi_ok']:.1f} \\\\")

print("=" * 80)
print("LaTeX rows for Table 6 (paste into sn-article.tex):")
print("=" * 80)
for cell in valid_cells:
    a = aggregate_per_cell(results, cell)
    print(latex_row(cell, "cp",   a["cp"],   bold=False))
    print(latex_row(cell, "pccp", a["pccp"], bold=True))
    print(latex_row(cell, "fcp",  a["fcp"],  bold=False))
    print("\\midrule")
avg_a = {}
for method in ["cp", "pccp", "fcp"]:
    avg_a[method] = {
        "PICP_mean": float(np.mean([r[method]["PICP"] for r in results])),
        "PICP_std":  float(np.std([r[method]["PICP"] for r in results])),
        "MPIW_mean": float(np.mean([r[method]["MPIW"] for r in results])),
        "MPIW_std":  float(np.std([r[method]["MPIW"] for r in results])),
        "psi_neg":   float(np.mean([r[method]["psi_neg"] for r in results])),
        "psi_pos":   float(np.mean([r[method]["psi_pos"] for r in results])),
        "psi_ok":    float(np.mean([r[method]["psi_ok"] for r in results])),
    }
print(latex_row("Average", "cp",   avg_a["cp"]))
print(latex_row("Average", "pccp", avg_a["pccp"], bold=True))
print(latex_row("Average", "fcp",  avg_a["fcp"]))
